# Self-supervised pretraining with AstroLens AstroPT

A tutorial-scale pretraining example for `astrolens.models.astropt.AstroPT`
(Smith et al., 2024, https://arxiv.org/abs/2405.14930), using
[`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10)
(Galaxy10 DECals, 17,736 galaxies).

Unlike `Linformer` and `GCNN`, AstroPT is a self-supervised backbone: it
learns from unlabeled images alone, with a causal next-patch prediction
objective (`model.loss(images)`). This notebook trains a small model from
scratch on GZ10 to demonstrate the mechanism and saves a checkpoint.

For a downstream classification task, see
[`gz10_astropt_finetuning.ipynb`](gz10_astropt_finetuning.ipynb), which
LoRA-finetunes a released, much larger pretrained checkpoint from
[`Smith42/astroPT`](https://huggingface.co/Smith42/astroPT) instead of the
tiny from-scratch model trained here — five epochs on 17k images is enough to
see the objective decrease, not to reach a genuinely useful backbone.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [ ]:
!pip install -q datasets torchvision

## Imports

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import astrolens
from utils import gz10

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset

Pretraining needs images only, no labels: a plain 90/10 train/val split.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64

data, train_idx, val_idx = gz10.load_split_9010()

train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)

train_dataset = gz10.GZ10Dataset(data, train_idx, train_transform, with_label=False)
val_dataset = gz10.GZ10Dataset(data, val_idx, eval_transform, with_label=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset)

## Create the model

A small config (`patch_size=16` -> 196 patches at 224x224, `dim=192`,
`depth=6`, `heads=3`) so a few epochs run in reasonable time on one GPU.
`num_classes` is left unset: `forward()` then returns next-patch predictions,
and `loss()` computes the autoregressive (Huber) pretraining objective.

In [ ]:
PATCH_SIZE = 16
DIM = 192
DEPTH = 6
HEADS = 3

model = astrolens.create_model(
    "astropt",
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
).to(device)

sum(p.numel() for p in model.parameters())

## Pretrain

In [ ]:
PRETRAIN_EPOCHS = 5
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, count = 0.0, 0
    with torch.set_grad_enabled(train):
        for images in loader:
            images = images.to(device)
            loss = model.loss(images)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            count += images.size(0)

    return total_loss / count


for epoch in range(1, PRETRAIN_EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

## Save the checkpoint

In [ ]:
torch.save(
    {
        "model": model.state_dict(),
        "img_size": IMG_SIZE,
        "patch_size": PATCH_SIZE,
        "dim": DIM,
        "depth": DEPTH,
        "heads": HEADS,
    },
    "astropt_gz10_pretrained.pt",
)